# Setup

In [ ]:
import duckdb
from splink import Linker, DuckDBAPI, SettingsCreator, block_on
import splink.comparison_library as cl
import splink.comparison_level_library as cll

con = duckdb.connect()

df_l = con.read_parquet("./perturbed_data_l.parquet")
df_r = con.read_parquet("./perturbed_data_r.parquet")

db_api = DuckDBAPI(con)

df_l = db_api.register(df_l, table_name="df_l")
df_r = db_api.register(df_r, table_name="df_r")

In [ ]:
# look at data
con.sql("select * from df_l")

# Defining comparisons

In [ ]:
# get dataset count
N_l = con.sql("select count(*) as n from df_l").fetchall()[0][0]

In [ ]:
settings = SettingsCreator(
    link_type="link_and_dedupe",
    # define your comparisons
    comparisons=[
        
    ],
    # copy in your blocking rules from previous notebook
    blocking_rules_to_generate_predictions=[
        
    ],
    # come up with a sensible probability or estimate this value
    probability_two_random_records_match=1/1000,
    additional_columns_to_retain=["person_id", "family_id"],
    retain_intermediate_calculation_columns=True,
    retain_matching_columns=True,
)

linker = Linker([df_l, df_r], settings, log_level=20)

# Estimating parameters

In [ ]:
linker.training.estimate_u_using_random_sampling(
    max_pairs=1e7,
)

In [ ]:
# add more EM training sessions as needed
ts1 = linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("dob"),
    estimate_without_term_frequencies=True,
)

model = linker.misc.save_model_to_json("./model.json", overwrite=True)

In [ ]:
c = linker.visualisations.parameter_estimate_comparisons_chart()
c.save_offline_chart("./parameter_estimate_comparisons_chart.html", overwrite=True)
c

In [ ]:
c = linker.visualisations.match_weights_chart()
c.save_offline_chart("./match_weights_chart.html", overwrite=True)
c

# Create predictions

In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.1)

In [ ]:
linker.visualisations.comparison_viewer_dashboard(
    df_predict,
    out_path="./comparison_viewer.html",
    overwrite=True,
    num_example_rows=10,
    minimum_comparison_vector_count=1,
)